In [83]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from time import *
import pandas as pd
import os
from tensorflow.keras.callbacks import CSVLogger
import matplotlib.pyplot as plt

print(tf.__version__)

2.17.0


# Masuk ke Directory Dataset_Final_Kfold

In [10]:
cd dataset_final_kfold/

C:\Users\gusma\Learn Jupyter Notebook\mobileNetV2\Training MobileNetV2\Kfold\dataset_final_kfold


# Melihat Directory saat ini
## Harus sudah didalam folder dataset kfold ya!

In [58]:
pwd

'C:\\Users\\gusma\\Learn Jupyter Notebook\\mobileNetV2\\Training MobileNetV2\\Kfold\\dataset_final_kfold'

# Membuat dataframe

In [72]:
import os 

directories = os.listdir()

filename = []
label = []


for i in range(0, len(directories)):
    os.chdir(directories[i])
    print(os.getcwd())
    files = os.listdir()
    for j in range(0, len(files)):
        files[j] = directories[i] + "/" + files[j]
        label.append(directories[i])
        print(files[j])
    filename = filename + files
    os.chdir("..")
print(filename)
print(label)


C:\Users\gusma\Learn Jupyter Notebook\mobileNetV2\Training MobileNetV2\Kfold\dataset_final_kfold\class1
class1/putih1.jpg
class1/putih10.jpg
class1/putih100.jpg
class1/putih101.jpg
class1/putih102.jpg
class1/putih103.jpg
class1/putih104.jpg
class1/putih105.jpg
class1/putih106.jpg
class1/putih107.jpg
class1/putih108.jpg
class1/putih109.jpg
class1/putih11.jpg
class1/putih110.jpg
class1/putih111.jpg
class1/putih112.jpg
class1/putih113.jpg
class1/putih114.jpg
class1/putih115.jpg
class1/putih116.jpg
class1/putih117.jpg
class1/putih118.jpg
class1/putih119.jpg
class1/putih12.jpg
class1/putih120.jpg
class1/putih121.jpg
class1/putih122.jpg
class1/putih124.jpg
class1/putih125.jpg
class1/putih126.jpg
class1/putih127.jpg
class1/putih128.jpg
class1/putih129.jpg
class1/putih13.jpg
class1/putih130.jpg
class1/putih131.jpg
class1/putih132.jpg
class1/putih133.jpg
class1/putih134.jpg
class1/putih135.jpg
class1/putih136.jpg
class1/putih138.jpg
class1/putih139.jpg
class1/putih14.jpg
class1/putih140.jpg
cla

In [73]:
print(len(filename))

2340


In [74]:
# save list of image to pandas
import pandas as pd

# key + value
data = {"filename": filename, "label":label}

df = pd.DataFrame(data)

# print dataframe
print(df)

                 filename   label
0       class1/putih1.jpg  class1
1      class1/putih10.jpg  class1
2     class1/putih100.jpg  class1
3     class1/putih101.jpg  class1
4     class1/putih102.jpg  class1
...                   ...     ...
2335    class9/emas94.jpg  class9
2336    class9/emas95.jpg  class9
2337    class9/emas97.jpg  class9
2338    class9/emas98.jpg  class9
2339    class9/emas99.jpg  class9

[2340 rows x 2 columns]


In [75]:
# acak dataframe yang dibuat
df_suffled = df.sample(frac=1)
print(df_suffled)

                         filename    label
160            class1/putih78.jpg   class1
2040         class8/coklat156.jpg   class8
1536      class5/hijaumuda200.jpg   class5
209       class10/hijautua128.jpg  class10
124            class1/putih39.jpg   class1
...                           ...      ...
1445      class5/hijaumuda103.jpg   class5
1900          class7/birutua2.jpg   class7
1042  class2/putih_campuran53.jpg   class2
322        class10/hijautua55.jpg  class10
509           class11/hitam66.jpg  class11

[2340 rows x 2 columns]


In [76]:
# cross validation
from sklearn.model_selection import KFold

# train data sudah berbentuk array excel(tabel)
train_data = df_suffled # sudah diacak
# train_data = df # tanpa di acak

# Y adalah label sebenarnya
Y= train_data['label']

print(len(Y.value_counts()))

kf = KFold(n_splits=9)

13


In [77]:
# ini data input harus diisi
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [85]:
cd ..

C:\Users\gusma\Learn Jupyter Notebook\mobileNetV2\Training MobileNetV2\Kfold


In [86]:
# import checkpoint
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import ModelCheckpoint
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator


image_dir = 'C:\\Users\\gusma\\Learn Jupyter Notebook\\mobileNetV2\\dataset_final_kfold'

fold_var = 1

for train_index, val_index in kf.split(np.zeros(len(Y)), Y):


      # ambil data train 0-80 | validation 81-100 ditentukan oleh index dengan 5 kali contoh jika kfold 5
    training_data = train_data.iloc[train_index]
    validation_data = train_data.iloc[val_index]


    # membaca dari data frame bukan data directory | data frame yang sudah kita buat sebelumnya
    train_data_generator = train_datagen.flow_from_dataframe(
        training_data, directory=image_dir, x_col='filename', y_col='label',
        target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)


    valid_data_generator = valid_datagen.flow_from_dataframe(
      validation_data, directory=image_dir, x_col='filename', y_col='label',
      target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)


    print("Fold -", fold_var)

    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    
    # semua training ulang
    base_model.trainable = False
    
    # Menambahkan Global Average Pooling
    x = GlobalAveragePooling2D()(base_model.output)
    
    # Menambahkan lapisan Dense dengan aktivasi Softmax untuk klasifikasi 13 kelas
    output_layer = Dense(13, activation='softmax')(x)
    
    #mobilenetV2(Feature Map) dan Output layer FC menggunakan softmax
    model = Model(inputs=base_model.input, outputs=output_layer)
    
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    csv_logger = CSVLogger(f'training_log_fold_{fold_var}.csv', append=True)
    
    hist = model.fit(train_data_generator, validation_data = valid_data_generator,epochs=32, callbacks=[csv_logger], verbose=1)
    
    fold_var += 1



Found 2080 validated image filenames belonging to 13 classes.
Found 260 validated image filenames belonging to 13 classes.
Fold - 1
Epoch 1/5


C:\Users\gusma\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


65/65 ━━━━━━━━━━━━━━━━━━━━ 31s 411ms/step - accuracy: 0.5850 - loss: 1.5099 - val_accuracy: 0.9846 - val_loss: 0.1972
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 371ms/step - accuracy: 0.9904 - loss: 0.1515 - val_accuracy: 0.9962 - val_loss: 0.0881
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 373ms/step - accuracy: 0.9982 - loss: 0.0701 - val_accuracy: 0.9962 - val_loss: 0.0564
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 370ms/step - accuracy: 0.9993 - loss: 0.0436 - val_accuracy: 0.9962 - val_loss: 0.0402
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 373ms/step - accuracy: 1.0000 - loss: 0.0301 - val_accuracy: 0.9962 - val_loss: 0.0330
Found 2080 validated image filenames belonging to 13 classes.
Found 260 validated image filenames belonging to 13 classes.
Fold - 2
Epoch 1/5


C:\Users\gusma\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


65/65 ━━━━━━━━━━━━━━━━━━━━ 30s 391ms/step - accuracy: 0.5880 - loss: 1.4354 - val_accuracy: 0.9731 - val_loss: 0.1965
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 25s 376ms/step - accuracy: 0.9865 - loss: 0.1509 - val_accuracy: 1.0000 - val_loss: 0.0850
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 25s 374ms/step - accuracy: 0.9978 - loss: 0.0688 - val_accuracy: 1.0000 - val_loss: 0.0553
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 368ms/step - accuracy: 0.9999 - loss: 0.0409 - val_accuracy: 1.0000 - val_loss: 0.0402
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 373ms/step - accuracy: 1.0000 - loss: 0.0292 - val_accuracy: 1.0000 - val_loss: 0.0318
Found 2080 validated image filenames belonging to 13 classes.
Found 260 validated image filenames belonging to 13 classes.
Fold - 3
Epoch 1/5


...

65/65 ━━━━━━━━━━━━━━━━━━━━ 31s 415ms/step - accuracy: 0.5431 - loss: 1.6198 - val_accuracy: 0.9808 - val_loss: 0.2083
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 369ms/step - accuracy: 0.9913 - loss: 0.1521 - val_accuracy: 0.9962 - val_loss: 0.0892
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 25s 374ms/step - accuracy: 0.9977 - loss: 0.0700 - val_accuracy: 0.9962 - val_loss: 0.0563
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 368ms/step - accuracy: 0.9984 - loss: 0.0435 - val_accuracy: 1.0000 - val_loss: 0.0404
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 372ms/step - accuracy: 0.9988 - loss: 0.0314 - val_accuracy: 1.0000 - val_loss: 0.0298
Found 2080 validated image filenames belonging to 13 classes.
Found 260 validated image filenames belonging to 13 classes.
Fold - 8
Epoch 1/5


C:\Users\gusma\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


65/65 ━━━━━━━━━━━━━━━━━━━━ 29s 387ms/step - accuracy: 0.5824 - loss: 1.5164 - val_accuracy: 0.9885 - val_loss: 0.2045
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 366ms/step - accuracy: 0.9908 - loss: 0.1558 - val_accuracy: 0.9962 - val_loss: 0.0882
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 370ms/step - accuracy: 0.9969 - loss: 0.0748 - val_accuracy: 0.9962 - val_loss: 0.0534
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 367ms/step - accuracy: 0.9992 - loss: 0.0447 - val_accuracy: 0.9962 - val_loss: 0.0377
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 25s 374ms/step - accuracy: 1.0000 - loss: 0.0306 - val_accuracy: 1.0000 - val_loss: 0.0281
Found 2080 validated image filenames belonging to 13 classes.
Found 260 validated image filenames belonging to 13 classes.
Fold - 9
Epoch 1/5


C:\Users\gusma\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


65/65 ━━━━━━━━━━━━━━━━━━━━ 32s 421ms/step - accuracy: 0.5186 - loss: 1.6315 - val_accuracy: 0.9808 - val_loss: 0.2202
Epoch 2/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 369ms/step - accuracy: 0.9913 - loss: 0.1539 - val_accuracy: 0.9962 - val_loss: 0.0975
Epoch 3/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 26s 400ms/step - accuracy: 0.9975 - loss: 0.0746 - val_accuracy: 1.0000 - val_loss: 0.0603
Epoch 4/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 371ms/step - accuracy: 0.9973 - loss: 0.0435 - val_accuracy: 1.0000 - val_loss: 0.0449
Epoch 5/5
65/65 ━━━━━━━━━━━━━━━━━━━━ 24s 370ms/step - accuracy: 0.9994 - loss: 0.0287 - val_accuracy: 1.0000 - val_loss: 0.0352


# Konversi file ke TFLITE

In [82]:
# function untuk konversi tflite
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return size

def convert_bytes(size, unit=None):
    if unit == "KB":
        return print('File size ' + str(round(size / 1024, 3)) + 'Kilobytes')
    elif unit == "MB":
        return print('File size ' + str(round(size / 1024 * 1024, 3)) + 'Megabytes')
    else :
        return print('File size ' + str(round(size)) + 'bytes')

KERAS_MODEL_NAME = "model_epoch50_32.h5"
model.save(KERAS_MODEL_NAME)
convert_bytes(get_file_size(KERAS_MODEL_NAME), "MB")

# conversi menjadi tflite
TF_LITE_MODEL_FILE_NAME = "model_epoch50_32.tflite"

tf_lite_converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = tf_lite_converter.convert()

tflite_model_name = TF_LITE_MODEL_FILE_NAME
open(tflite_model_name, "wb").write(tflite_model)
convert_bytes(get_file_size(TF_LITE_MODEL_FILE_NAME), "KB")



File size 9753728.0Megabytes
INFO:tensorflow:Assets written to: C:\Users\gusma\AppData\Local\Temp\tmp2v81yqqw\assets


INFO:tensorflow:Assets written to: C:\Users\gusma\AppData\Local\Temp\tmp2v81yqqw\assets


Saved artifact at 'C:\Users\gusma\AppData\Local\Temp\tmp2v81yqqw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_2808')
Output Type:
  TensorSpec(shape=(None, 13), dtype=tf.float32, name=None)
Captures:
  2514524640784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2510302822224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2514524639632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2514524639824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2514524640592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2514524640016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2510297983952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2514355158672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2510297973200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2510297973776: TensorSpec(shape=(), dtype=tf.resource, name=None)
 